In [18]:
import logging
import sys
from pathlib import Path
import pandas as pd
import joblib


logging.basicConfig(
    format="%(asctime)s.%(msecs)d %(levelname)s %(filename)s:%(lineno)d %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger()
logger.setLevel(logging.INFO)


# Move up one level from scripts/ to find the project root
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

logging.info(f"Project root added to path: {PROJECT_ROOT}")

from src.core.dataset_parser import parse_claudette_zipfile
from src.core.feature_engine import SimpleEmbeddingEngine
from src.core.classifier import ClauseClassifier, MODELS_TO_EXPERIMENT




23:24:30.198 INFO 1351323694.py:21 Project root added to path: /home/miguel/enhesa-tos-service


In [19]:
artifacts_exp="simple_embeddings"
#artifacts_exp="legal_bert_embeddings"

zip_filepath = PROJECT_ROOT / "data" / "ToS.zip"
artifacts_dir = PROJECT_ROOT / "models" / artifacts_exp
models_output_dir = artifacts_dir / "classifiers"


In [ ]:
for model_name in MODELS_TO_EXPERIMENT.keys():
    model_folder_path = models_output_dir / model_name
    clause_clf = ClauseClassifier()
    clause_clf.load_model(model_folder_path)

    report_file_path = model_folder_path / "evaluation_report.joblib"
    metrics_payload = joblib.load(report_file_path)


    rows = []

    for model_name in MODELS_TO_EXPERIMENT.keys():
        model_folder_path = models_output_dir / model_name
        
        # Skip if the directory doesn't exist yet
        if not model_folder_path.exists():
            continue
            
        report_file_path = model_folder_path / "evaluation_report.joblib"
        metrics_payload = joblib.load(report_file_path)
        
        # 1. Build a clean dictionary for this row
        row_data = {
            "Model": model_name,
            "Macro F1": metrics_payload.get("macro_f1"),
            "Precision": metrics_payload.get("macro_precision"),
            "Recall": metrics_payload.get("macro_recall"),
            "Accuracy": metrics_payload.get("accuracy"),
            # Stringify the hyperparameters so they fit nicely into a single cell
        }
        
        rows.append(row_data)

    # ensemble
    model_name="MetaEnsemble_Stacking"
    model_folder_path = models_output_dir / model_name
    
        
    report_file_path = model_folder_path / "evaluation_report.joblib"
    metrics_payload = joblib.load(report_file_path)
    
    # 1. Build a clean dictionary for this row
    row_data = {
        "Model": model_name,
        "Macro F1": metrics_payload.get("macro_f1"),
        "Precision": metrics_payload.get("macro_precision"),
        "Recall": metrics_payload.get("macro_recall"),
        "Accuracy": metrics_payload.get("accuracy"),
        # Stringify the hyperparameters so they fit nicely into a single cell
    }
    
    rows.append(row_data)

# 2. Convert the collected rows into a structured Pandas DataFrame
df_results = pd.DataFrame(rows)




for model_name in MODELS_TO_EXPERIMENT.keys():
    model_folder_path = models_output_dir / model_name

    report_file_path = model_folder_path / "evaluation_report.joblib"
    metrics_payload = joblib.load(report_file_path)

    print(str(metrics_payload.get("best_params")))

23:24:30.221 INFO classifier.py:102 No estimator specified. Defaulting to Balanced Logistic Regression.
23:24:30.223 INFO classifier.py:169 Model framework successfully restored from /home/miguel/enhesa-tos-service/models/simple_embeddings/classifiers/LogisticRegression/clf_estimator.joblib
23:24:30.224 INFO classifier.py:102 No estimator specified. Defaulting to Balanced Logistic Regression.
23:24:30.225 INFO classifier.py:169 Model framework successfully restored from /home/miguel/enhesa-tos-service/models/simple_embeddings/classifiers/LogisticRegression/clf_estimator.joblib
23:24:30.226 INFO classifier.py:102 No estimator specified. Defaulting to Balanced Logistic Regression.
23:24:30.227 INFO classifier.py:169 Model framework successfully restored from /home/miguel/enhesa-tos-service/models/simple_embeddings/classifiers/LinearSVC/clf_estimator.joblib
23:24:30.228 INFO classifier.py:102 No estimator specified. Defaulting to Balanced Logistic Regression.
23:24:30.248 INFO classifier.

                  Model  Macro F1  Precision    Recall  Accuracy
0    LogisticRegression  0.727416   0.691178  0.842342  0.848115
1             LinearSVC  0.731772   0.694674  0.846260  0.851301
2      SVC_RBF_Pipeline  0.854551   0.864273  0.845508  0.944769
3          RandomForest  0.752610   0.754814  0.750462  0.904408
4  HistGradientBoosting  0.714960   0.855354  0.664238  0.917685
5         MLPClassifier  0.835156   0.874236  0.805397  0.941583
{'C': 10.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 5000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 42, 'solver': 'saga', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
{'C': 1.0, 'class_weight': 'balanced', 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 5000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.01, 'verbose': 0}
{'memory': None, 'steps': [('scaler', StandardSc

In [ ]:
print(df_results)